# Preparar y estructurar datos de Instacart con Python y Pandas
**GA1-220501094-01-AA1-EV06** · Aprendiz: David Vargas Franco · Ficha: 3169901

**Pregunta orientadora:** ¿Cómo preparar y estructurar los datos de Instacart para obtener información confiable sobre los hábitos de compra?

Este cuaderno desarrolla, en orden, las lecciones 2, 3 y 4 del material *Preparar y estructurar datos con Python*. Cada lección incluye los ejemplos, el laboratorio guiado y el reto. Como cada lección usa las variables de la anterior, el cuaderno debe ejecutarse de arriba abajo (*Kernel → Restart Kernel and Run All Cells*).

| Parte | Contenido |
|---|---|
| Preparación | Rutas y carga de los cinco CSV |
| Lección 2 | Limpieza y estandarización; Laboratorio guiado 2; Reto 2 |
| Lección 3 | Filtrado, ordenamiento y agrupación; Laboratorio guiado 3; Reto 3 |
| Lección 4 | Combinación de datasets; Laboratorio guiado 4; Reto 4 |

> **Rutas:** el material usa carpetas en español; este proyecto usa `data/raw` y `data/processed` (nombres en inglés), como pide la actividad.

## Preparación: rutas y carga de los datos

In [1]:
import pandas as pd
from pathlib import Path

RUTA = Path('../data/raw')            # el cuaderno vive en notebooks/
SALIDA = Path('../data/processed')    # archivos generados
SALIDA.mkdir(parents=True, exist_ok=True)

print(pd.__version__)

3.1.0


In [2]:
# Los archivos usan punto y coma como separador (sep=';')
orders      = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
order_prods = pd.read_csv(RUTA / 'order_products.csv', sep=';')
products    = pd.read_csv(RUTA / 'products.csv', sep=';')
aisles      = pd.read_csv(RUTA / 'aisles.csv', sep=';')
departments = pd.read_csv(RUTA / 'departments.csv', sep=';')

for nombre, df in [('orders', orders), ('order_prods', order_prods), ('products', products),
                   ('aisles', aisles), ('departments', departments)]:
    print(nombre, df.shape)

orders (478967, 6)
order_prods (4545007, 4)
products (49694, 4)
aisles (134, 2)
departments (21, 2)


Tamaños esperados: `orders (478967, 6)`, `order_prods (4545007, 4)`, `products (49694, 4)`, `aisles (134, 2)` y `departments (21, 2)`.

**Problemas detectados al explorar (Lección 1), que se resuelven en la Lección 2:** separador `;`, 15 pedidos duplicados, nulos en tres columnas (`days_since_prior_order`, `product_name`, `add_to_cart_order`), columnas enteras leídas como `float64` y la categoría centinela `missing`.

In [3]:
print(orders.isna().sum(), '\n')
print(products.isna().sum(), '\n')
print(order_prods.isna().sum(), '\n')
print('Pedidos duplicados:', orders.duplicated().sum(), '| order_id único:', orders['order_id'].is_unique)

order_id                      0
user_id                       0
order_number                  0
order_dow                     0
order_hour_of_day             0
days_since_prior_order    28819
dtype: int64 

product_id          0
product_name     1258
aisle_id            0
department_id       0
dtype: int64 

order_id               0
product_id             0
add_to_cart_order    836
reordered              0
dtype: int64 

Pedidos duplicados: 15 | order_id único: False


---
# Lección 2: Limpieza y estandarización de datos crudos

Para cada problema hay tres opciones: **eliminar, corregir o conservar**, y la elección depende de qué significa el dato. Orden de trabajo: duplicados, tipos, nulos, texto, duplicados implícitos y verificación.

## 2.1 Duplicados exactos: `duplicated` y `drop_duplicates`

In [4]:
duplicados = orders[orders.duplicated(keep=False)].sort_values('order_id')
print(len(duplicados))
duplicados[['order_id', 'order_dow', 'order_hour_of_day']].head(6)

30


,order_id,order_dow,order_hour_of_day
354993,391768,3,2
371905,391768,3,2
119251,408114,3,2
321100,408114,3,2
394347,467134,3,2
250626,467134,3,2


Se esperan **30 filas** (15 pedidos y sus 15 copias). Todas ocurren en la misma franja (`order_dow = 3`, `order_hour_of_day = 2`: miércoles a las 2 a. m.), lo que apunta a un fallo del sistema y no a pedidos reales. **Decisión: eliminar.**

In [5]:
orders = orders.drop_duplicates().reset_index(drop=True)
print(orders.shape, orders['order_id'].is_unique)

(478952, 6) True


## 2.2 Tipos de dato con `astype`

In [6]:
print('Antes:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

orders = orders.astype({
    'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
    'order_dow': 'int8', 'order_hour_of_day': 'int8',
    'days_since_prior_order': 'Int16',
})
order_prods = order_prods.astype({
    'order_id': 'int32', 'product_id': 'int32',
    'add_to_cart_order': 'Int16', 'reordered': 'int8',
})

print('Después:', round(orders.memory_usage(deep=True).sum() / 1e6, 1), 'MB')

Antes: 23.0 MB
Después: 7.2 MB


Se esperan `Antes: 23.0 MB` y `Después: 7.2 MB`. `Int16` (con mayúscula) es un tipo *nullable*: admite enteros y nulos a la vez; con `int16` minúscula habría un `IntCastingNaNError`.

## 2.3 Nulos: comprobar la causa antes de decidir

In [7]:
nulos = orders['days_since_prior_order'].isna()
primeros = orders['order_number'] == 1
print((nulos == primeros).all())

True


Se espera `True`: cada nulo es un primer pedido (no existe pedido anterior). **Decisión: conservar el nulo**; rellenar con 0 inventaría que el cliente volvió el mismo día y bajaría la media de espera.

In [8]:
print(products[products['product_name'].isna()][['aisle_id', 'department_id']].value_counts())
products['product_name'] = products['product_name'].fillna('Unknown')
print(products['product_name'].isna().sum())

aisle_id  department_id
100       21               1258
Name: count, dtype: int64
0


Se espera una sola combinación (`100` y `21`, es decir, pasillo y departamento `missing`) con 1 258 productos, y luego `0` nulos. **Decisión: corregir** con un valor explícito, porque el producto existe y no se puede borrar.

| Columna | Causa | Decisión |
| --- | --- | --- |
| `days_since_prior_order` | Primer pedido del cliente | Conservar el nulo |
| `product_name` | Producto sin datos de catálogo | Rellenar con `'Unknown'` |
| `add_to_cart_order` | Se investiga en el Reto 2 | Conservar el nulo |

## 2.4 Texto con los métodos `.str`

In [9]:
nombres = products['product_name']
print('Espacios dobles:', nombres.str.contains('  ', regex=False).sum())
print('Espacio invisible:', nombres.str.contains('\xa0', regex=False).sum())

products['product_name'] = (
    products['product_name']
    .str.replace('\xa0', ' ', regex=False)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
    .str.title()
)
print(products['product_name'].str.contains('  ', regex=False).sum(),
      products['product_name'].str.contains('\xa0', regex=False).sum())

Espacios dobles: 403
Espacio invisible: 16
0 0


Se esperan `Espacios dobles: 403`, `Espacio invisible: 16` y, tras limpiar, `0 0`. El espacio invisible (`\xa0`) se reemplaza en un paso propio porque en pandas 3 la expresión `\s` no lo reconoce.

## 2.5 Duplicados implícitos

In [10]:
conocidos = products[products['product_name'] != 'Unknown']
repetidos = conocidos[conocidos['product_name'].duplicated(keep=False)]
print(conocidos['product_name'].duplicated().sum())
print(repetidos['product_id'].isin(order_prods['product_id']).sum(), 'de', len(repetidos))

141
268 de 280


Se esperan `141` nombres repetidos y `268 de 280` productos implicados que sí aparecen en pedidos. **Decisión: conservar y documentar.** Tienen `product_id` distintos y borrarlos dejaría pedidos apuntando a productos inexistentes. En los análisis se agrupa por `product_id`, no por nombre.

## 2.6 Limpieza reutilizable con funciones y verificación

In [11]:
def limpiar_orders(df):
    df = df.drop_duplicates().reset_index(drop=True)
    df = df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                    'order_dow': 'int8', 'order_hour_of_day': 'int8',
                    'days_since_prior_order': 'Int16'})
    assert df['order_id'].is_unique
    return df

def limpiar_products(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

print(limpiar_orders(orders).shape)

(478952, 6)


In [12]:
assert orders.shape == (478952, 6)
assert orders['days_since_prior_order'].isna().sum() == 28817   # 2 nulos estaban entre los duplicados
assert products['product_name'].isna().sum() == 0
assert str(order_prods['add_to_cart_order'].dtype) == 'Int16'
print('Limpieza verificada')

Limpieza verificada


**Resumen de la Lección 2 (ejemplos):** duplicados exactos (`duplicated`, `drop_duplicates`), tipos (`astype`, tipos *nullable*), causa de los nulos (`fillna` o conservar), estandarización de texto (`.str`), duplicados implícitos y verificación con `assert`.

## Laboratorio guiado 2: pipeline de limpieza con informe de cambios

Un **pipeline** encadena pasos de limpieza, se ejecuta con una sola instrucción y deja un **informe** de lo que cambió en cada paso. Se trabaja sobre copias nuevas de los archivos crudos.

### Paso 1: recargar los datos crudos

In [13]:
orders_crudo   = pd.read_csv(RUTA / 'instacart_orders.csv', sep=';')
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
op_crudo       = pd.read_csv(RUTA / 'order_products.csv', sep=';')
print(orders_crudo.shape, products_crudo.shape, op_crudo.shape)

(478967, 6) (49694, 4) (4545007, 4)


### Paso 2: función que anota el estado de la tabla
`anotar` devuelve la tabla **sin cambios**, por eso se puede intercalar entre pasos.

In [14]:
registro = []

def anotar(df, tabla, paso):
    registro.append({
        'tabla': tabla,
        'paso': paso,
        'filas': len(df),
        'nulos': int(df.isna().sum().sum()),
        'memoria_mb': round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })
    return df

### Paso 3: un paso de limpieza por función

In [15]:
def quitar_duplicados(df):
    return df.drop_duplicates().reset_index(drop=True)

def tipos_orders(df):
    return df.astype({'order_id': 'int32', 'user_id': 'int32', 'order_number': 'int16',
                      'order_dow': 'int8', 'order_hour_of_day': 'int8',
                      'days_since_prior_order': 'Int16'})

def limpiar_nombres(df):
    df = df.copy()
    df['product_name'] = (df['product_name'].fillna('Unknown')
                          .str.replace('\xa0', ' ', regex=False)
                          .str.replace(r'\s+', ' ', regex=True)
                          .str.strip().str.title())
    return df

def tipos_order_prods(df):
    return df.astype({'order_id': 'int32', 'product_id': 'int32',
                      'add_to_cart_order': 'Int16', 'reordered': 'int8'})

### Paso 4: encadenar los pasos con `pipe`

In [16]:
orders_limpio = (orders_crudo
    .pipe(anotar, 'orders', '0 crudo')
    .pipe(quitar_duplicados)
    .pipe(anotar, 'orders', '1 sin duplicados')
    .pipe(tipos_orders)
    .pipe(anotar, 'orders', '2 tipos'))

products_limpio = (products_crudo
    .pipe(anotar, 'products', '0 crudo')
    .pipe(limpiar_nombres)
    .pipe(anotar, 'products', '1 nombres'))

op_limpio = (op_crudo
    .pipe(anotar, 'order_prods', '0 crudo')
    .pipe(tipos_order_prods)
    .pipe(anotar, 'order_prods', '1 tipos'))

### Paso 5: el informe de cambios

In [17]:
informe = pd.DataFrame(registro)
informe

,tabla,paso,filas,nulos,memoria_mb
0,orders,0 crudo,478967,28819,23.0
1,orders,1 sin duplicados,478952,28817,23.0
2,orders,2 tipos,478952,28817,7.2
3,products,0 crudo,49694,1258,3.1
4,products,1 nombres,49694,0,3.1
5,order_prods,0 crudo,4545007,836,145.4
6,order_prods,1 tipos,4545007,836,54.5


**Interpretación del informe (con mis resultados):**

- Quitar duplicados eliminó 15 filas de `orders` (478.967 → 478.952) y 2 nulos (28.819 → 28.817): dos de los pedidos repetidos eran primeros pedidos.
- Corregir tipos no cambió filas ni nulos, pero redujo la memoria de `orders` de 23,0 a 7,2 MB y la de `order_prods` de 145,4 a 54,5 MB.
- Estandarizar los nombres dejó `products` sin nulos (1.258 → 0) al rellenar con `Unknown`.
- Los 836 nulos de `order_prods` siguen a propósito: se decidió conservarlos (ver Reto 2, ejercicio 3).


### Paso 6: validar el resultado

In [18]:
def validar(o, p, op):
    assert o['order_id'].is_unique, 'order_id repetido'
    assert o['order_hour_of_day'].between(0, 23).all(), 'hora fuera de rango'
    assert p['product_name'].notna().all(), 'productos sin nombre'
    assert not p['product_name'].str.contains('\xa0', regex=False).any(), 'espacios invisibles'
    assert op['reordered'].isin([0, 1]).all(), 'reordered con valores raros'
    return 'Validación superada'

print(validar(orders_limpio, products_limpio, op_limpio))
print(orders_limpio.equals(orders), products_limpio.equals(products))

Validación superada
True True


Se esperan `Validación superada` y `True True`: el pipeline produce las mismas tablas que la limpieza manual de los ejemplos.

**Conclusiones clave:** un pipeline es una cadena de funciones pequeñas que reciben y devuelven un DataFrame; intercalar una función de registro da un informe de cambios sin esfuerzo extra; y validar con `assert` convierte las reglas de calidad en una prueba que se ejecuta cada vez.

### Guardar los resultados de limpieza

In [19]:
informe.to_csv(SALIDA / 'cleaning_report.csv', index=False)
orders_limpio.to_parquet(SALIDA / 'orders_clean.parquet', index=False)
products_limpio.to_parquet(SALIDA / 'products_clean.parquet', index=False)
print(sorted(p.name for p in SALIDA.iterdir()))

['.gitkeep', 'cleaning_report.csv', 'orders_clean.parquet', 'products_clean.parquet']


## Reto 2

Cada ejercicio tiene su código y una respuesta. **Completa las respuestas con tu propia salida.**

### Ejercicio 1: duplicados y valores de `order_prods`

In [20]:
print('Filas duplicadas exactas:', order_prods.duplicated().sum())
print('Pares order_id + product_id repetidos:', order_prods.duplicated(subset=['order_id', 'product_id']).sum())
print('Valores de reordered:', sorted(order_prods['reordered'].unique().tolist()))

Filas duplicadas exactas: 0
Pares order_id + product_id repetidos: 0
Valores de reordered: [0, 1]


**Respuesta:** `order_prods` no tiene filas duplicadas exactas (0), ningún par `order_id` + `product_id` se repite (0) y `reordered` solo contiene los valores 0 y 1. No requiere limpieza de duplicados ni de valores.

### Ejercicio 2: nombres de `aisles` y `departments`

In [21]:
for nombre, df, col in [('aisles', aisles, 'aisle'), ('departments', departments, 'department')]:
    s = df[col]
    print(nombre,
          '| nombres duplicados:', s.duplicated().sum(),
          '| espacios al inicio o al final:', (s != s.str.strip()).sum(),
          '| espacios dobles:', s.str.contains('  ', regex=False).sum())

aisles | nombres duplicados: 0 | espacios al inicio o al final: 0 | espacios dobles: 0
departments | nombres duplicados: 0 | espacios al inicio o al final: 0 | espacios dobles: 0


**Respuesta:** en `aisles` y en `departments` hay 0 nombres duplicados, 0 con espacios al inicio o al final y 0 con espacios dobles, así que **no necesitan limpieza de texto**. El valor `missing` no es un error de formato: es la categoría centinela que se trata en el ejercicio 7.

### Ejercicio 3: nulos de `add_to_cart_order`

In [22]:
sin_posicion = order_prods[order_prods['add_to_cart_order'].isna()]
pedidos_afectados = sin_posicion['order_id'].unique()
print('Nulos:', len(sin_posicion), '| pedidos afectados:', len(pedidos_afectados))

tamano = (order_prods[order_prods['order_id'].isin(pedidos_afectados)]
          .groupby('order_id').size())
print('Productos por pedido afectado: mínimo', tamano.min(), '| máximo', tamano.max())
print('Máximo valor registrado en add_to_cart_order:', order_prods['add_to_cart_order'].max())

Nulos: 836 | pedidos afectados: 70
Productos por pedido afectado: mínimo 65 | máximo 127
Máximo valor registrado en add_to_cart_order: 64


**Respuesta:** hay 836 nulos repartidos en **70 pedidos**. Cada pedido afectado tiene como mínimo 65 productos (el mayor tiene 127) y el máximo valor registrado en `add_to_cart_order` es 64. La explicación es que la columna **solo registra posiciones hasta la 64**: los nulos son los productos que van después en pedidos muy grandes. No son errores aleatorios, por eso se conservan; solo hay que recordar que de esos productos se desconoce el orden en que se agregaron.

### Ejercicio 4: nombres que cambian con la estandarización

In [23]:
products_crudo = pd.read_csv(RUTA / 'products.csv', sep=';')
limpios = limpiar_nombres(products_crudo)['product_name']
cambiaron = (products_crudo['product_name'].notna() & (products_crudo['product_name'] != limpios)).sum()
print('Nombres (no nulos) que cambiaron:', cambiaron)

Nombres (no nulos) que cambiaron: 6321


**Respuesta:** cambiaron **6.321 nombres** (alrededor del 13 % de los 48.436 nombres no nulos). Solo 403 tenían espacios dobles y 16 un espacio invisible, así que la mayoría cambió probablemente por la normalización de mayúsculas (`.title()`) y de espacios en los extremos. Estandarizar evita contar como productos distintos nombres que solo difieren en el formato.

### Ejercicio 5: categoría ordenada para el día de la semana

In [24]:
dias_orden = ['domingo', 'lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado']
dia_texto = orders['order_dow'].map(dict(enumerate(dias_orden)))
dia_cat = dia_texto.astype(pd.CategoricalDtype(dias_orden, ordered=True))

print(dia_cat.dtype)
print('Texto:', round(dia_texto.memory_usage(deep=True) / 1e6, 2), 'MB')
print('Categoría:', round(dia_cat.memory_usage(deep=True) / 1e6, 2), 'MB')
dia_cat.value_counts().sort_index()

category
Texto: 7.14 MB
Categoría: 0.48 MB


order_dow
domingo      84090
lunes        82185
martes       65833
miércoles    60897
jueves       59810
viernes      63488
sábado       62649
Name: count, dtype: int64

**Respuesta:** como texto la columna ocupa **7,14 MB** y como categoría **0,48 MB**, unas 15 veces menos. La categoría guarda cada día como un código pequeño y la lista de nombres una sola vez. Además, al ser ordenada, `sort_index()` muestra los días de domingo a sábado y no por orden alfabético.

### Ejercicio 6: productos `'Unknown'` en los pedidos

In [25]:
ids_unknown = products.loc[products['product_name'] == 'Unknown', 'product_id']
filas_unknown = order_prods['product_id'].isin(ids_unknown)
print('Filas de order_prods con productos Unknown:', filas_unknown.sum())
print('Pedidos distintos que los incluyen:', order_prods.loc[filas_unknown, 'order_id'].nunique())

Filas de order_prods con productos Unknown: 9547
Pedidos distintos que los incluyen: 8254


**Respuesta:** **9.547 filas** de `order_prods` (0,21 % del total) corresponden a productos `Unknown` y aparecen en **8.254 pedidos distintos**. **No los eliminaría**: son compras reales (cada fila es un producto que alguien pidió) y borrarlas reduciría el tamaño de los pedidos y sesgaría los análisis. Lo adecuado es conservarlos, marcarlos como `Unknown` y, cuando se analicen productos, mostrarlos aparte.

### Ejercicio 7: `'missing'` como nulo real en una copia de `departments`

In [26]:
dept_copia = departments.copy()
dept_copia['department'] = dept_copia['department'].mask(dept_copia['department'] == 'missing')
print(dept_copia.tail(3))
print('Nulos en la copia:', dept_copia['department'].isna().sum())

    department_id department
18             19     snacks
19             20       deli
20             21        NaN
Nulos en la copia: 1


**Respuesta:** no conviene hacerlo en la tabla original antes de agrupar porque `groupby` **descarta por defecto las filas con clave nula**: los productos del departamento `missing` desaparecerían de los resultados sin ningún aviso, en lugar de aparecer como un grupo visible.

## Documentación de la limpieza (Lección 2)

| Problema | Evidencia | Decisión |
|---|---|---|
| Duplicados exactos en `orders` | 15 pedidos repetidos (30 filas, todas del miércoles a las 2 a. m.) | **Eliminar** → 478.952 pedidos |
| Tipos de dato | Memoria de `orders`: 23,0 → 7,2 MB; de `order_prods`: 145,4 → 54,5 MB | **Corregir** con enteros pequeños y `Int16` |
| Nulos en `days_since_prior_order` | 28.817, que son los primeros pedidos de cada cliente | **Conservar** el nulo |
| Nulos en `product_name` | 1.258, todos del pasillo 100 y departamento 21 (`missing`) | **Corregir**: rellenar con `Unknown` |
| Nulos en `add_to_cart_order` | 836 en 70 pedidos de 65 o más productos; la columna llega hasta 64 | **Conservar** |
| Texto sin estandarizar | 403 nombres con espacios dobles y 16 con espacio invisible; 6.321 nombres cambian al estandarizar | **Corregir** con `.str` |
| Duplicados implícitos | 141 nombres repetidos con `product_id` distinto; 268 de 280 productos implicados se piden | **Conservar** y agrupar por `product_id` |
| Productos `Unknown` | 9.547 filas en 8.254 pedidos | **Conservar** y mostrarlos aparte |
| Categoría centinela `missing` | Departamento 21 y pasillo 100 | **Conservar** en las tablas; tratar como nulo solo en análisis concretos |

Los resultados se guardaron en `data/processed/`: `cleaning_report.csv`, `orders_clean.parquet` y `products_clean.parquet`.

---
## Cierre de la Lección 2
Limpieza documentada en `data/processed/cleaning_report.csv`, `orders_clean.parquet` y `products_clean.parquet`. Las Lecciones 3 y 4 continúan en este mismo cuaderno.